In [1]:
## Packages to be installed- ansys_fluent_core, bayesian_optimization

import ansys.fluent.core as pyfluent
import os
import bayes_opt
from bayes_opt import BayesianOptimization, UtilityFunction

In [2]:
## Function to run each ANSYS FLUENT Simulation
def run_ansys_sim(lmbda,power,eta_0,eta_inf):
    
    global count
    
    ## eta_inf chosen by optimizer always has to be less than eta_0
    if eta_inf > eta_0:
        return 0
    
    ## Can change fluent version number, number of processesors and whether to show GUI or not
    session = pyfluent.launch_fluent("22.2.0",precision="double",version="2d", processor_count=4, mode="solver", show_gui=True)
    tui = session.tui
    tui.file.read_case("Racetracking.cas.h5") ## These would change depending 
    tui.file.read_data("Racetracking.dat.h5") ## on simulation Case file used

    ## Stores log for each simulation, can use it to check for error and fill time. 
    filename="Simulation_Log"+str(count)
    tui.file.start_transcript(filename)

    ## Changes the carreau model to incorporate new set of parameters
    tui.define.materials.change_create("resin","resin","no","no","no","yes","carreau","shear-rate-dependent",lmbda,power,eta_0,eta_inf)
    tui.solve.initialize.initialize_flow()
    tui.solve.dual_time_iterate(6000,50,"no","yes")
    

    # Checks for Floating Point Error/ Divergence in Simulation log and alters URF accordingly to re-run simulation
    with open(filename, 'r') as f:
        last_line = f.readlines()[-1]
        if "Error Object: #f" in last_line:
            tui.solve.set.under_relaxation("pressure",0.3,"density",1,"body-force",1,"mom",0.7,"mp",0.5)
            tui.solve.initialize.initialize_flow()
            tui.solve.dual_time_iterate(6000,50,"no","yes")
    f.close()

    # Contour object has to be created in Case file, this will simply save the image generated at end of fill
    tui.display.objects.display("fill-contour")
    filename="Fill_Contour"+str(count)
    tui.display.save_picture(filename)

    # Target function (Resin Vf) is stored in the last line of the report file, configured in Case file itself. 
    with open('Output.txt', 'r') as f:
        last_line = f.readlines()[-1]    
        outputvf = float((last_line.split())[1]) ## Output Variable
        f.close()

    count = count + 1

    ## Force quits simulation due to FLUENT Not Responding error 
    tui.exit("yes")
    os.system("taskkill /f /im cx2220.exe")
    
    return outputvf

In [4]:
## Change parameter sample space here

optimizer = BayesianOptimization(f=None, pbounds={'lmbda':[1e-4,1e5],'power':[1e-4,1],'eta_0':[1e-2,1e2],
                                                  'eta_inf':[1e-3,1e2]}, verbose=2, random_state=1234)

In [4]:
## Change value of kappa here for optimization, xi plays no role (kappa=10k signifies more exploration)

utility = UtilityFunction(kind='ucb',kappa=10000,xi=10000)

In [5]:
## Function to add previously obtained simulation data .txt file to optimizer before next set of iterations.

def AcquireValues(filename):
    with open(filename,'r') as f:
        dat = f.read()
    dat = dat.split('\n')
    dat = dat[1:-1]
    for i,j in enumerate(dat):
        dat[i] = [float(x) for x in j.split('\t')]
    return dat

## Combine all Parameters_Archive.txt files from previous simulations into a file called Parameters_Archive_Prior.txt

data = AcquireValues('Parameters_Archive_Prior.txt')
key_set = ['lmbda','power','eta_0','eta_inf']
for i in data:
    next_values = {}
    for key_idx,key in enumerate(key_set):
        next_values[key] = i[key_idx]
    target = i[4]
    optimizer.register(params = next_values,target=target)

In [5]:
global count
count=1

## Creates .txt file showing all the parameters sampled.
with open('Parameters_Archive.txt','w') as f:
    f.writelines("lambda \t power \t eta_0 \t eta_inf \t target \n")
    f.close()
    
## CHANGE THIS VALUE TO CHANGE NUMBER OF ITERATIONS RUN!!!!!!    
while count < 101:
    next_values = optimizer.suggest(utility)
    with open('Parameters_Archive.txt','a') as f:
        par = str(next_values['lmbda']) + '\t' + str(next_values['power']) + '\t' + str(next_values['eta_0']) + '\t' + str(next_values['eta_inf']) + '\t'
        f.writelines(par)
        f.close()
    
    target = run_ansys_sim(**next_values)
    
    with open('Parameters_Archive.txt','a') as f:
        f.writelines(str(target) + '\n')
        f.close()
    
    try:
        optimizer.register(params = next_values,target=target)
    except:
        print('Got some errors')

Fast-loading "C:\PROGRA~1\ANSYSI~1\v222\fluent\fluent22.2.0\\addons\afd\lib\hdfio.bin"
Done.

EN-MA-SWSN-38 is already loaded (40.4559).                       Process affinity not being set.

Reading from EN-MA-SWSN-38:"C:\Users\rg776\Desktop\Bayesian Optimization\Racetracking.cas.h5" in NODE0 mode ...
  Reading mesh ...
        7989 cells,     2 cell zones ...
           2560 mixed cells,  zone id: 3
           5429 mixed cells,  zone id: 4
       16210 faces,     6 face zones ...
           4711 2D interior faces,  zone id: 1
          10707 2D interior faces,  zone id: 2
            165 2D pressure-inlet faces,  zone id: 7
             17 2D pressure-outlet faces,  zone id: 8
            296 2D interior faces,  zone id: 9
            314 2D wall faces,  zone id: 10
        8222 nodes,     1 node zone  ...
  Done.


Building...
     mesh
	distributing mesh
		parts....,
		faces....,
		nodes....,
		cells....,
        bandwidth reduction using Reverse Cuthill-McKee: 1639/58 = 28.2586
  

    74  1.4400e+00  1.2236e-03  2.2423e-03  1.0473e-02  0:00:00   26
    75  1.4285e+00  1.1966e-03  2.2117e-03  1.0438e-02  0:00:00   25
    76  1.4141e+00  1.1721e-03  2.1763e-03  1.0497e-02  0:00:00   24
    77  1.3993e+00  1.1517e-03  2.1495e-03  1.0485e-02  0:00:00   23
    78  1.3856e+00  1.1296e-03  2.1172e-03  1.0529e-02  0:00:00   22
    79  1.3787e+00  1.1093e-03  2.0875e-03  1.0590e-02  0:00:00   21
    80  1.3693e+00  1.0934e-03  2.0642e-03  1.0628e-02  0:00:00   20
    81  1.3710e+00  1.0802e-03  2.0408e-03  1.0724e-02  0:00:00   19
    82  1.3501e+00  1.0665e-03  2.0167e-03  1.0727e-02  0:00:00   18

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
    83  1.3641e+00  1.0558e-03  1.9954e-03  1.0817e-02  0:00:00   17
    84  1.3496e+00  1.0475e-03  1.9806e-03  1.0864e-02  0:00:00   16
    85  1.3431e+00  1.0449e-03  1.9650e-03  1.0954e-02  0:00:00   15
    86  1.3273e+00  1.0395e-03  1.9575e-03  1.1030e-02  0:00:00   14
    87  1.2989e+00  1.0353e-03  1

   182  1.5787e+00  9.0837e-04  1.8953e-03  7.6311e-03  0:00:00   18

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
   183  1.5636e+00  8.9582e-04  1.8663e-03  7.3666e-03  0:00:00   17
   184  1.6016e+00  8.9698e-04  1.8329e-03  7.5155e-03  0:00:00   16
   185  1.8080e+00  8.8850e-04  1.8173e-03  7.2355e-03  0:00:00   15
   186  1.8346e+00  8.8138e-04  1.7852e-03  6.9953e-03  0:00:03   14
   187  1.8213e+00  8.8907e-04  1.8084e-03  7.1704e-03  0:00:02   13
   188  1.9449e+00  8.9192e-04  1.7925e-03  6.8703e-03  0:00:02   12
   189  2.0051e+00  8.8019e-04  1.7611e-03  6.7574e-03  0:00:01   11
   190  2.0522e+00  8.7270e-04  1.7681e-03  6.8265e-03  0:00:01   10
   191  2.1143e+00  8.6741e-04  1.7500e-03  6.5495e-03  0:00:01    9
   192  2.1136e+00  8.5569e-04  1.6950e-03  6.5372e-03  0:00:00    8
   193  2.2153e+00  8.5016e-04  1.7066e-03  6.5209e-03  0:00:00    7

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
   194  2.4238e+00  8.4407e-04  

   284  6.5755e-01  5.0255e-04  1.3858e-03  1.1368e-03  0:00:03   16
   285  7.0133e-01  4.9128e-04  1.3615e-03  1.0945e-03  0:00:02   15
   286  7.7256e-01  4.9463e-04  1.3592e-03  1.0698e-03  0:00:01   14
   287  8.4872e-01  4.8313e-04  1.3688e-03  1.0122e-03  0:00:01   13
   288  9.5711e-01  4.8159e-04  1.3405e-03  9.6800e-04  0:00:01   12
   289  1.0354e+00  4.8167e-04  1.3720e-03  9.5087e-04  0:00:01   11
   290  1.1021e+00  4.7597e-04  1.3416e-03  9.0480e-04  0:00:00   10
   291  1.1229e+00  4.7094e-04  1.3412e-03  8.5594e-04  0:00:00    9
   292  1.0119e+00  4.6994e-04  1.3355e-03  8.4656e-04  0:00:00    8
   293  1.0050e+00  4.5900e-04  1.2979e-03  8.0789e-04  0:00:00    7

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
   294  1.0147e+00  4.5752e-04  1.3049e-03  7.5448e-04  0:00:00    6
   295  9.9998e-01  4.5207e-04  1.3070e-03  7.3893e-04  0:00:00    5
   296  9.7677e-01  4.5275e-04  1.3062e-03  7.1344e-04  0:00:00    4
   297  9.5958e-01  4.4630e-04  1

   386  2.1260e-01  3.3475e-04  1.2369e-03  1.3659e-04  0:00:00   14
   387  2.0790e-01  3.3528e-04  1.2381e-03  1.2879e-04  0:00:03   13
   388  2.0458e-01  3.2771e-04  1.2258e-03  1.2109e-04  0:00:02   12
   389  2.0130e-01  3.2549e-04  1.2033e-03  1.1230e-04  0:00:01   11
   390  1.9672e-01  3.1799e-04  1.1840e-03  1.0535e-04  0:00:01   10
   391  1.9459e-01  3.1142e-04  1.1709e-03  9.9187e-05  0:00:01    9
   392  1.9621e-01  3.1138e-04  1.1719e-03  9.2356e-05  0:00:01    8
   393  1.9283e-01  3.0954e-04  1.1813e-03  8.7634e-05  0:00:00    7

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
   394  1.8825e-01  3.0538e-04  1.1793e-03  8.2481e-05  0:00:00    6
   395  1.8075e-01  3.0649e-04  1.2005e-03  7.6650e-05  0:00:00    5
   396  1.7091e-01  3.0112e-04  1.1792e-03  7.1375e-05  0:00:00    4
   397  1.6688e-01  2.9680e-04  1.1420e-03  6.7485e-05  0:00:00    3
   398  1.6254e-01  2.9460e-04  1.1643e-03  6.4222e-05  0:00:00    2
   399  1.6415e-01  2.9355e-04  1

...

  2977  1.1189e-03  2.4331e-05  2.2675e-05  1.7509e-06  0:00:05   23
  2978  1.0175e-03  2.2439e-05  2.0482e-05  1.5475e-06  0:00:04   22
  2979  9.2463e-04  2.0681e-05  1.8503e-05  1.3648e-06  0:00:03   21
  2980  8.3960e-04  1.9035e-05  1.6709e-05  1.2077e-06  0:00:02   20
  2981  7.5440e-04  1.7578e-05  1.5097e-05  1.0760e-06  0:00:02   19
  2982  6.8650e-04  1.6132e-05  1.3655e-05  9.6189e-07  0:00:01   18

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
  2983  6.2441e-04  1.4776e-05  1.2361e-05  8.6077e-07  0:00:01   17
  2984  5.6751e-04  1.3489e-05  1.1190e-05  7.7600e-07  0:00:01   16
  2985  5.1509e-04  1.2330e-05  1.0144e-05  7.0852e-07  0:00:01   15
  2986  4.6925e-04  1.1302e-05  9.2005e-06  6.5290e-07  0:00:00   14
  2987  4.2852e-04  1.0309e-05  8.3503e-06  6.0427e-07  0:00:00   13
  2988  3.9266e-04  9.3966e-06  7.5862e-06  5.5396e-07  0:00:00   12
  2989  3.5999e-04  8.5577e-06  6.8944e-06  5.0831e-07  0:00:00   11
  2990  3.2986e-04  7.7899e-06  6

  3085  3.9773e-04  9.7084e-06  8.3110e-06  7.7272e-07  0:00:00   15
  3086  3.6530e-04  8.8245e-06  7.5286e-06  7.2467e-07  0:00:00   14
  3087  3.3594e-04  8.0129e-06  6.8257e-06  6.7957e-07  0:00:00   13
  3088  3.0980e-04  7.2713e-06  6.1961e-06  6.3852e-07  0:00:00   12
  3089  2.8603e-04  6.5923e-06  5.6330e-06  6.0018e-07  0:00:00   11
  3090  2.6430e-04  5.9701e-06  5.1279e-06  5.6384e-07  0:00:00   10
  3091  2.4465e-04  5.4013e-06  4.6749e-06  5.2912e-07  0:00:00    9
  3092  2.2698e-04  4.8843e-06  4.2693e-06  4.9595e-07  0:00:00    8
  3093  2.1081e-04  4.4147e-06  3.9064e-06  4.6468e-07  0:00:00    7

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
  3094  1.9578e-04  3.9889e-06  3.5811e-06  4.3419e-07  0:00:00    6
  3095  1.8362e-04  3.3983e-06  3.1623e-06  4.0936e-07  0:00:00    5
  3096  1.7118e-04  3.2765e-06  3.0472e-06  3.8183e-07  0:00:00    4
  3097  1.5931e-04  2.8041e-06  2.7083e-06  3.6102e-07  0:00:00    3
  3098  1.4887e-04  2.7079e-06  2

  3191  1.8020e-04  4.3608e-06  3.6094e-06  2.9798e-07  0:00:00    9
  3192  1.6588e-04  3.9232e-06  3.2834e-06  2.6854e-07  0:00:00    8
  3193  1.5283e-04  3.3113e-06  2.8692e-06  2.5224e-07  0:00:00    7

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
  3194  1.3833e-04  3.2448e-06  2.7967e-06  2.2817e-07  0:00:00    6
  3195  1.2875e-04  2.7173e-06  2.4439e-06  2.1537e-07  0:00:00    5
  3196  1.1707e-04  2.6391e-06  2.3753e-06  1.9547e-07  0:00:00    4
  3197  1.0888e-04  2.2178e-06  2.0826e-06  1.8496e-07  0:00:00    3
  3198  1.0163e-04  2.1156e-06  1.9995e-06  1.6925e-07  0:00:00    2
  3199  9.2941e-05  1.8123e-06  1.7777e-06  1.6053e-07  0:00:00    1
  3200  8.4636e-05  1.7679e-06  1.7599e-06  1.4697e-07  0:00:00    0
(if (< 0.95 (pick-a-real "report/surface-integrals/area-weighted-avg/2,/resin/vof/no"))(set! mstop? #t))#f


> 
Flow time = 6.4s, time step = 64
5936 more time steps

Updating solution at time levels N and N-1.
 done.

  iter  continuity  x

  3298  6.7144e-05  1.3949e-06  1.4268e-06  1.7140e-07  0:00:00    2
  3299  6.2435e-05  1.2577e-06  1.3118e-06  1.6274e-07  0:00:00    1
  3300  5.7473e-05  1.1866e-06  1.2632e-06  1.5541e-07  0:00:00    0
(if (< 0.95 (pick-a-real "report/surface-integrals/area-weighted-avg/2,/resin/vof/no"))(set! mstop? #t))#f


> 
Flow time = 6.600000000000001s, time step = 66
5934 more time steps

Updating solution at time levels N and N-1.
 done.

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
  3300  5.7473e-05  1.1866e-06  1.2632e-06  1.5541e-07  0:00:00   50
  3301  5.2868e-05  1.0314e-06  1.1460e-06  1.4368e-04  0:00:00   49
  3302  8.5260e-04  2.7263e-06  9.4267e-06  1.1213e-04  0:00:00   48
  3303  1.5109e-03  5.4918e-06  1.5929e-05  8.8096e-05  0:00:00   47
  3304  1.3916e-03  1.1113e-05  2.3011e-05  6.9601e-05  0:00:00   46
  3305  1.6859e-03  1.5926e-05  2.8619e-05  5.4955e-05  0:00:00   45
  3306  1.8524e-03  2.0356e-05  3.2901e-05  4.3394e-05  0:00:00   44
  3307  

  3399  6.2429e-05  8.9506e-07  9.8445e-07  8.5396e-08  0:00:00    1
  3400  5.8370e-05  8.1083e-07  9.1119e-07  7.9777e-08  0:00:00    0
(if (< 0.95 (pick-a-real "report/surface-integrals/area-weighted-avg/2,/resin/vof/no"))(set! mstop? #t))#f


> 
Flow time = 6.800000000000001s, time step = 68
5932 more time steps

Updating solution at time levels N and N-1.
 done.

  iter  continuity  x-velocity  y-velocity    vf-resin     time/iter
  3400  5.8370e-05  8.1083e-07  9.1119e-07  7.9777e-08  0:00:01   50
  3401  5.4454e-05  7.4331e-07  8.6118e-07  9.7533e-05  0:00:01   49
  3402  3.5111e-04  1.8151e-06  5.8064e-06  7.6292e-05  0:00:01   48
  3403  5.5956e-04  3.8757e-06  1.0165e-05  5.9909e-05  0:00:01   47
  3404  7.0136e-04  6.5991e-06  1.4174e-05  4.6984e-05  0:00:01   46
  3405  7.9326e-04  9.5325e-06  1.7527e-05  3.6829e-05  0:00:00   45
  3406  8.5727e-04  1.2176e-05  2.0288e-05  2.8955e-05  0:00:09   44
  3407  8.8606e-04  1.4110e-05  2.1540e-05  2.2758e-05  0:00:07   43
  3408  

In [6]:
## Prints the best results from the entire optimizer registry as a float number

print("Best Results: {}; Airvf: {:.6f}".format(optimizer.max['target'],optimizer.max['target']))

Best Results: 0.9988963326392831; Airvf: 0.998896
